# AzSemBench — STS Fine-Tuning


Fine-tunes four multilingual Transformer **cross-encoders** on Azerbaijani STS:
- XLM-RoBERTa-base
- multilingual BERT
- multilingual DistilBERT
- mDeBERTa-v3-base

The model sees both sentences jointly and predicts a score in the 0–5 STS range.

Model evaluation:
Pearson, Spearman, Kendall tau, MSE, RMSE, MAE, median absolute error, R²,
explained variance, and max error.

The notebook also computes the agreed lexical baselines:
Levenshtein, Jaro-Winkler, SequenceMatcher, token Jaccard, token Dice,
token overlap coefficient, char-3 Jaccard, and char-4 Jaccard.

Best checkpoint is selected on **dev Pearson**. Test is held out from model selection.



Run top to bottom in Colab, Kaggle (Internet + GPU enabled), or local Jupyter. Outputs are written under `outputs/`. See the repository README for metrics, supplied results and dataset filtering.


## Runtime setup
Enable Internet and a GPU in Kaggle/Colab. The notebook downloads its dataset from GitHub; no manual dataset upload is needed. Models train sequentially. Edit `MODELS` to run a subset.


In [ ]:
%pip install -q "pandas>=2.2,<3" "numpy>=1.26,<3" "scipy>=1.12,<2" "scikit-learn>=1.4,<2" "sentence-transformers>=3.4,<6" "transformers>=4.48,<5" "datasets>=3,<5" "accelerate>=1,<2" "torch>=2.4,<3" "sentencepiece>=0.2,<1" "protobuf>=4,<7" "rapidfuzz>=3,<4" "python-dotenv>=1,<2"


### Tokenizer compatibility fix

This version tries the fast tokenizer first and automatically falls back to `use_fast=False` if the checkpoint/runtime cannot instantiate a fast tokenizer. The slow tokenizer is valid for fine-tuning; it is only somewhat slower during preprocessing.


In [ ]:
from pathlib import Path
from difflib import SequenceMatcher
import gc, inspect, re, shutil, time
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from datasets import Dataset
from rapidfuzz.distance import Levenshtein, JaroWinkler
from scipy.stats import pearsonr, spearmanr, kendalltau
from sklearn.metrics import mean_squared_error, mean_absolute_error, median_absolute_error, r2_score, explained_variance_score, max_error
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding, Trainer, TrainingArguments, EarlyStoppingCallback, set_seed

print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
import os
MODELS = [
    "FacebookAI/xlm-roberta-base",
    "google-bert/bert-base-multilingual-cased",
    "distilbert/distilbert-base-multilingual-cased",
    "microsoft/mdeberta-v3-base",
]
SEED=42
MAX_LENGTH=192
TRAIN_BATCH_SIZE=8
EVAL_BATCH_SIZE=32
GRAD_ACCUM_STEPS=2
LEARNING_RATE=2e-5
WEIGHT_DECAY=0.01
EPOCHS=5
EARLY_STOPPING_PATIENCE=2
OUTPUT_DIR=Path(os.getenv("AZSEMBENCH_OUTPUT_DIR", "outputs/sts_finetuned"))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
set_seed(SEED)


In [ ]:
# Configuration: environment variables override these defaults.
import os
from pathlib import Path
from urllib.request import urlretrieve
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass
REPO = os.getenv("AZSEMBENCH_REPO", "MehribanAliyeva/azsembench")
REF = os.getenv("AZSEMBENCH_REF", "main")  # use a commit SHA for a fixed dataset
DATA_URL = f"https://raw.githubusercontent.com/{REPO}/{REF}/data/model_ready/core_model_ready.csv"
DATA_DIR = Path("data/model_ready")
DATA_DIR.mkdir(parents=True, exist_ok=True)
CSV_PATH = DATA_DIR / "core_model_ready.csv"
# Download on each run so data is explicitly tied to REPO/REF.
try:
    urlretrieve(DATA_URL, CSV_PATH)
except Exception as exc:
    raise RuntimeError(f"Cannot download {DATA_URL}. Publish the prepared repo first and enable Internet.") from exc
import hashlib
print("Dataset:", DATA_URL)
print("SHA256:", hashlib.sha256(CSV_PATH.read_bytes()).hexdigest())

def clean(x):
    return "" if x is None or pd.isna(x) else str(x).strip()

csv_path = CSV_PATH
df=pd.read_csv(CSV_PATH,dtype=object,keep_default_na=False,encoding="utf-8-sig")
if "model_ready" in df.columns:
    df=df[df["model_ready"].astype(str).str.lower().isin(["true","1","yes"])].copy()

mask=pd.Series(False,index=df.index)
if "task" in df.columns:
    mask |= df["task"].astype(str).str.lower().eq("semantic_similarity")
if "recomputed_task" in df.columns:
    mask |= df["recomputed_task"].astype(str).str.lower().eq("semantic_similarity")
if "source_dataset" in df.columns:
    mask |= df["source_dataset"].astype(str).str.lower().str.contains("sts",regex=False)

sts=df.loc[mask].copy()
for c in ["sentence1_az","sentence2_az","label","split"]:
    if c not in sts.columns:
        raise RuntimeError(f"Missing {c}")
sts["sentence1_az"]=sts["sentence1_az"].map(clean)
sts["sentence2_az"]=sts["sentence2_az"].map(clean)
sts["label"]=pd.to_numeric(sts["label"],errors="raise").astype(float)
sts["split"]=sts["split"].astype(str).str.lower().str.strip()
sts=sts[(sts["sentence1_az"]!="")&(sts["sentence2_az"]!="")].copy()

train_df=sts[sts["split"].eq("train")].copy()
dev_df=sts[sts["split"].eq("dev")].copy()
test_df=sts[sts["split"].eq("test")].copy()
print(sts["split"].value_counts())

for split_name, frame in [("train", train_df), ("dev", dev_df), ("test", test_df)]:
    if frame.empty: raise ValueError(f"Empty {split_name} split")
if not sts["label"].between(0, 5).all(): raise ValueError("STS labels must lie in [0,5].")


In [ ]:
def corr3(y,p):
    y=np.asarray(y,float); p=np.asarray(p,float)
    if len(y)<2 or np.std(y)==0 or np.std(p)==0:
        return np.nan,np.nan,np.nan
    return float(pearsonr(y,p).statistic),float(spearmanr(y,p).statistic),float(kendalltau(y,p).statistic)

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, balanced_accuracy_score, matthews_corrcoef

def reg_metrics(y,p):
    y=np.asarray(y,float); p=np.asarray(p,float)
    pear,spear,kend=corr3(y,p)
    mse=float(mean_squared_error(y,p))
    return {
        "pearson":pear,"spearman":spear,"kendall_tau":kend,
        "mse":mse,"rmse":float(np.sqrt(mse)),
        "mae":float(mean_absolute_error(y,p)),
        "median_absolute_error":float(median_absolute_error(y,p)),
        "r2":float(r2_score(y,p)),
        "explained_variance":float(explained_variance_score(y,p)),
        "max_error":float(max_error(y,p)),
        "binary_threshold":4.0,
        "binary_accuracy":float(accuracy_score(y>=4, p>=4)),
        "binary_precision":float(precision_score(y>=4, p>=4, zero_division=0)),
        "binary_recall":float(recall_score(y>=4, p>=4, zero_division=0)),
        "binary_f1":float(f1_score(y>=4, p>=4, zero_division=0)),
        "binary_balanced_accuracy":float(balanced_accuracy_score(y>=4, p>=4)),
        "binary_mcc":float(matthews_corrcoef(y>=4, p>=4)),
    }

def compute_metrics(eval_pred):
    logits,labels=eval_pred
    return reg_metrics(labels,np.asarray(logits).reshape(-1))

def safe_name(x):
    return re.sub(r"[^A-Za-z0-9_.-]+","__",x)

def training_args(out):
    kw = dict(
        output_dir=str(out),
        learning_rate=LEARNING_RATE,
        per_device_train_batch_size=TRAIN_BATCH_SIZE,
        per_device_eval_batch_size=EVAL_BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM_STEPS,
        num_train_epochs=EPOCHS,
        weight_decay=WEIGHT_DECAY,
        save_strategy="epoch",
        logging_strategy="steps",
        logging_steps=50,
        save_total_limit=1,
        load_best_model_at_end=True,
        metric_for_best_model="pearson",
        greater_is_better=True,
        report_to="none",
        seed=SEED,
        data_seed=SEED,
        fp16=False,
        bf16=False,
    )

    # Transformers/Kaggle compatibility:
    # only pass arguments supported by the installed version.
    valid_args = inspect.signature(TrainingArguments.__init__).parameters

    if "eval_strategy" in valid_args:
        kw["eval_strategy"] = "epoch"
    elif "evaluation_strategy" in valid_args:
        kw["evaluation_strategy"] = "epoch"

    kw = {k: v for k, v in kw.items() if k in valid_args}

    return TrainingArguments(**kw)


class Float32RegressionTrainer(Trainer):
    """
    Force the STS regression loss to be computed in float32.

    This avoids dtype-mismatch failures such as:
    RuntimeError: Found dtype Float but expected Half
    on Kaggle/Accelerate combinations that autocast model logits to float16.
    """
    def compute_loss(
        self,
        model,
        inputs,
        return_outputs=False,
        num_items_in_batch=None,
        **kwargs,
    ):
        inputs = dict(inputs)
        labels = inputs.pop("labels").float()

        outputs = model(**inputs)
        logits = outputs.logits.squeeze(-1).float()

        loss = F.mse_loss(logits, labels)

        return (loss, outputs) if return_outputs else loss

def make_ds(frame,tok):
    ds=Dataset.from_pandas(frame[["sentence1_az","sentence2_az","label"]].reset_index(drop=True))
    ds=ds.map(lambda b: tok(b["sentence1_az"],b["sentence2_az"],truncation=True,max_length=MAX_LENGTH),batched=True)
    ds=ds.rename_column("label","labels")
    keep={"input_ids","attention_mask","token_type_ids","labels"}
    return ds.remove_columns([c for c in ds.column_names if c not in keep])


def load_tokenizer_robust(model_name):
    """Try fast tokenizer; fall back to slow tokenizer if conversion fails."""
    try:
        tok = AutoTokenizer.from_pretrained(model_name, use_fast=True)
        print(f"Tokenizer for {model_name}: FAST")
        return tok
    except Exception as e:
        print(f"Fast tokenizer unavailable for {model_name}: {type(e).__name__}: {e}")
        print("Falling back to slow tokenizer...")
        tok = AutoTokenizer.from_pretrained(model_name, use_fast=False)
        print(f"Tokenizer for {model_name}: SLOW")
        return tok


In [ ]:
metric_rows=[]
pred_frames=[]
summary=[]

for model_name in MODELS:
    print("\\n","="*90,"\\n",model_name)
    out=OUTPUT_DIR/safe_name(model_name)
    tok=load_tokenizer_robust(model_name)
    tr=make_ds(train_df,tok); dv=make_ds(dev_df,tok); te=make_ds(test_df,tok)
    model=AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=1,
        problem_type="regression",
        ignore_mismatched_sizes=True,
        torch_dtype=torch.float32,
    )
    model = model.float()
    trainer_kwargs = dict(
        model=model,
        args=training_args(out),
        train_dataset=tr,
        eval_dataset=dv,
        data_collator=DataCollatorWithPadding(tok),
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE)],
    )

    # Transformers version compatibility:
    # older releases use `tokenizer=`, newer releases use `processing_class=`.
    trainer_params = inspect.signature(Trainer.__init__).parameters
    if "processing_class" in trainer_params:
        trainer_kwargs["processing_class"] = tok
    elif "tokenizer" in trainer_params:
        trainer_kwargs["tokenizer"] = tok

    trainer = Float32RegressionTrainer(**trainer_kwargs)
    t0=time.time()
    result=trainer.train()
    trainer.save_model(out/"best_model")
    tok.save_pretrained(out/"best_model")

    for split_name,ds,frame in [("train",tr,train_df),("dev",dv,dev_df),("test",te,test_df)]:
        raw=np.asarray(trainer.predict(ds).predictions).reshape(-1)
        clipped=np.clip(raw,0,5)
        for variant,p in [("raw",raw),("clipped_0_5",clipped)]:
            metric_rows.append({"model":model_name,"metric":"model_score","prediction_variant":variant,"split":split_name,"n":len(frame),**reg_metrics(frame["label"],p)})
        z=frame[["sentence1_az","sentence2_az","label","split"]].copy()
        if "source_id" in frame.columns:
            z.insert(0,"source_id",frame["source_id"].astype(str).values)
        z.insert(0,"model",model_name)
        z["prediction_raw"]=raw
        z["prediction_clipped_0_5"]=clipped
        pred_frames.append(z)

    summary.append({"model":model_name,"best_dev_pearson":trainer.state.best_metric,"best_checkpoint":trainer.state.best_model_checkpoint,"train_seconds":time.time()-t0,"train_loss":result.training_loss})
    pd.DataFrame(metric_rows).to_csv(OUTPUT_DIR/"sts_finetuned_all_metrics.csv", index=False)
    pd.concat(pred_frames, ignore_index=True).to_csv(OUTPUT_DIR/"sts_finetuned_predictions.csv", index=False)
    pd.DataFrame(summary).to_csv(OUTPUT_DIR/"sts_training_summary.csv", index=False)
    for checkpoint in out.glob("checkpoint-*"):
        if checkpoint.is_dir(): shutil.rmtree(checkpoint)
    del trainer,model,tok,tr,dv,te
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

model_metrics=pd.DataFrame(metric_rows)
predictions=pd.concat(pred_frames,ignore_index=True)
training_summary=pd.DataFrame(summary)
display(model_metrics[model_metrics["split"].eq("test")].sort_values("pearson",ascending=False))


In [ ]:
def toks(s): return [t for t in clean(s).lower().split() if t]
def ngrams(s,n):
    s=clean(s).lower()
    if not s: return set()
    return {s} if len(s)<n else {s[i:i+n] for i in range(len(s)-n+1)}
def jac(a,b):
    a,b=set(a),set(b)
    if not a and not b:return 1.0
    if not a or not b:return 0.0
    return len(a&b)/len(a|b)
def dice(a,b):
    a,b=set(a),set(b)
    if not a and not b:return 1.0
    return 2*len(a&b)/(len(a)+len(b)) if len(a)+len(b) else 0.0
def overlap(a,b):
    a,b=set(a),set(b)
    if not a and not b:return 1.0
    if not a or not b:return 0.0
    return len(a&b)/min(len(a),len(b))
def lex(s1,s2):
    t1,t2=toks(s1),toks(s2)
    return {
        "levenshtein_similarity":float(Levenshtein.normalized_similarity(s1,s2)),
        "jaro_winkler_similarity":float(JaroWinkler.normalized_similarity(s1,s2)),
        "sequence_matcher_ratio":float(SequenceMatcher(None,s1,s2).ratio()),
        "token_jaccard":jac(t1,t2),"token_dice":dice(t1,t2),"token_overlap_coefficient":overlap(t1,t2),
        "char3_jaccard":jac(ngrams(s1,3),ngrams(s2,3)),
        "char4_jaccard":jac(ngrams(s1,4),ngrams(s2,4)),
    }

lexdf=pd.DataFrame([lex(a,b) for a,b in zip(sts["sentence1_az"],sts["sentence2_az"])],index=sts.index)
lexrows=[]
for m in lexdf.columns:
    for split in ["train","dev","test","overall"]:
        idx=sts.index if split=="overall" else sts.index[sts["split"].eq(split)]
        p,s,k=corr3(sts.loc[idx,"label"],lexdf.loc[idx,m])
        lexrows.append({"model":"LEXICAL_BASELINE","metric":m,"prediction_variant":"surface_similarity","split":split,"n":len(idx),"pearson":p,"spearman":s,"kendall_tau":k,
                        "mse":np.nan,"rmse":np.nan,"mae":np.nan,"median_absolute_error":np.nan,"r2":np.nan,"explained_variance":np.nan,"max_error":np.nan})
lex_metrics=pd.DataFrame(lexrows)

final_metrics=pd.concat([model_metrics,lex_metrics],ignore_index=True)
final_metrics.to_csv(OUTPUT_DIR/"sts_finetuned_all_metrics.csv",index=False)
predictions.to_csv(OUTPUT_DIR/"sts_finetuned_predictions.csv",index=False)
training_summary.to_csv(OUTPUT_DIR/"sts_training_summary.csv",index=False)

lexout=sts[["sentence1_az","sentence2_az","label","split"]].copy()
for c in lexdf.columns: lexout[c]=lexdf[c].values
lexout.to_csv(OUTPUT_DIR/"sts_lexical_scores.csv",index=False)

display(final_metrics[final_metrics["split"].eq("test")].sort_values("pearson",ascending=False))
from IPython.display import FileLink
for path in sorted(OUTPUT_DIR.glob("*.csv")):
    display(FileLink(str(path)))
